In [ ]:
import pandas as pd
import numpy as np


In [ ]:

wishlist_path = "/home/chanyoung/GitHub/VIP_project/count NHIS data/NHIS Questionnaire_Cancer and LM (5).xlsx"
wishlist_df = pd.read_excel(wishlist_path, skiprows=1)

valid_vars = wishlist_df['Variable'].dropna()

target_variables = valid_vars.str.strip().tolist()

cancer_base_vars = [
    'BLADDCAN_A', 'BLOODCAN_A', 'BONECAN_A', 'BRAINCAN_A', 'BREASCAN_A', 'CERVICAN_A', 
    'COLRCCAN_A', 'ESOPHCAN_A', 'GALLBCAN_A', 
    'HDNCKCAN_A', 'LEUKECAN_A', 'LIVERCAN_A', 'LUNGCAN_A', 'LYMPHCAN_A', 'MELANCAN_A', 'OVARYCAN_A', 
    'PANCRCAN_A', 'PROSTCAN_A', 'SKNMCAN_A', 'SKNNMCAN_A', 'SKNDKCAN_A', 'STOMACAN_A', 
    'THYROCAN_A', 'UTERUCAN_A', 'OTHERCANP_A'
]


cancer_age_vars = [var.replace('CAN_A', 'AGETC_A').replace('CANP_A', 'AGETC_A') for var in cancer_base_vars]


for var in (cancer_base_vars + cancer_age_vars):
    if var not in target_variables:
        target_variables.append(var)

data_path = "adult24.csv"
nhis_df = pd.read_csv(data_path, low_memory=False)




In [ ]:
actual_columns = set(nhis_df.columns)

matched_vars = [var for var in target_variables if var in actual_columns]
missing_vars = [var for var in target_variables if var not in actual_columns]

print(f"Total number of target variables (including cancer vars): {len(target_variables)}")
print(f"Variables that successfully matched the dataset: {len(matched_vars)}")
if missing_vars:
    print(f"Matching failed (check for typos in name, etc.): {missing_vars}\n")

# 데이터 추출
extracted_df = nhis_df[matched_vars].copy()


cancer_counts = {}
for col in cancer_base_vars:
    if col in extracted_df.columns:
        cancer_counts[col] = (extracted_df[col] == 1).sum()

# 빈도수 기준으로 내림차순 정렬하여 가장 많은 암 찾기
sorted_cancers = sorted(cancer_counts.items(), key=lambda x: x[1], reverse=True)

# 상위 4개 암 변수명 추출
top_4_cancer_vars = [item[0] for item in sorted_cancers[:4]]
print(f"\n[데이터 기반 상위 4개 암종 (Top 4 Cancers)]")
for var, count in sorted_cancers[:4]:
    print(f" - {var}: {count}명")

# 나머지 암 변수명 추출 (Other Cancer로 합칠 대상들)
remaining_cancer_vars = [item[0] for item in sorted_cancers[4:]]

# 'Merged_Other_Cancer' 파생 변수 생성 (기본값 2 = No)
extracted_df['Merged_Other_Cancer'] = 2

# 나머지 암 변수 중 하나라도 1(Yes)인 경우 Merged_Other_Cancer를 1(Yes)로 덮어쓰기
# any(axis=1)을 사용하여 행 단위로 1이 하나라도 존재하는지 빠르게 체크
if remaining_cancer_vars:
    mask_other_cancer = (extracted_df[remaining_cancer_vars] == 1).any(axis=1)
    extracted_df.loc[mask_other_cancer, 'Merged_Other_Cancer'] = 1

# 'Merged_Other_Cancer_Age' 파생 변수 생성 (통합된 Other 암들의 진단 연령 중 '최솟값'을 추출)
remaining_age_vars = [var.replace('CAN_A', 'AGETC_A').replace('CANP_A', 'AGETC_A') for var in remaining_cancer_vars]
existing_remaining_age_vars = [var for var in remaining_age_vars if var in extracted_df.columns]

def get_min_valid_age(row):
    # NHIS 연령 데이터 중 97, 98, 99(모름, 거절 등)를 제외하고 실제 나이 데이터만 수집하여 최솟값 반환
    valid_ages = [age for age in row if pd.notna(age) and age < 97]
    return min(valid_ages) if valid_ages else np.nan

extracted_df['Merged_Other_Cancer_Age'] = extracted_df[existing_remaining_age_vars].apply(get_min_valid_age, axis=1)

# (선택 사항) 합쳐진 나머지 개별 암 변수들을 최종 CSV에서 제거 (깔끔한 데이터 유지를 위함)
columns_to_drop = remaining_cancer_vars + existing_remaining_age_vars
extracted_df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
### 🌟 새로 추가된 부분 끝 🌟 ###

# 5. 최종 CSV 저장
output_filename = "extracted_variables_with_merged_cancers.csv"
extracted_df.to_csv(output_filename, index=False)
W

In [ ]:


extracted_df = nhis_df[matched_vars]

output_filename = "extracted_variables.csv"
extracted_df.to_csv(output_filename, index=False)

print(f"\n{len(matched_vars)} successfully matched variable data points have been saved to the file '{output_filename}'!")